# Student Reflection RAG — Retrieval Step

## Building the Retrieval Index

* Load the synthetic data (synthetic_student_reflection_corpus.json)

* Embed each sample with the all-MiniLM-L6-v2.
  - This is a ONE-TIME batch embedding step that happens once at startup, not per-request. 
  
  - Same "load once" principle as loading the BERT model itself.

* Build a FAISS IndexFlatL2 from those embeddings.
  - This is the searchable structure retrieve_similar() will query against.

* retrieve_similar(text, k=3):
  - Embeds the NEW input text (not corpus — corpus is already embedded).

  - Searches the FAISS index for the 'k' closest corpus vectors by L2 distance.

  - FAISS returns INDEX POSITIONS (integers), not the vectors themselves or the text.

  - Those indices are used to look up the original plain-text dicts from example_corpus — this is what actually gets returned.

* Sanity check before trusting it on new text: retrieve using one of the corpus's own student_text entries as the query. 
  - The #1 result should be that exact same example (distance ≈ 0) — confirms the embed → index → search round-trip works correctly.

* What this step produces: a working retrieve_similar() function.
  - Nothing here touches BERT or Gemini yet — this is purely the (retrieval) in RAG, built and validated in isolation before wiring it into the full pipeline.

In [ ]:
# %pip install sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json 
from sentence_transformers import SentenceTransformer 
import faiss 
import numpy as np

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load synthetic data in
with open("synthetic_student_reflections.json") as f:
  example_corpus = json.load(f)

In [5]:
print(f"Loaded {len(example_corpus)} examples.")

Loaded 25 examples.


### Embed Student Texts

In [ ]:
# Load the embedder. 
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedder

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11024.40it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [9]:
# Get only the student_text from the JSON.
student_texts = [ex["student_text"] for ex in example_corpus]
student_texts[:3]

['I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.',
 'My group accidentally submitted the draft with all of our comments still visible. We were embarrassed at first, but looking back at some of the comments is actually pretty funny.',
 'I spent three hours working on the assignment and then found out the instructions had changed after I started. I wish someone had told us earlier because now I have to redo most of my work.']

In [11]:
# Embed all student_text entries. 
student_text_embeddings = embedder.encode(student_texts, convert_to_numpy=True)
student_text_embeddings[:5]

array([[-0.08167081, -0.0350743 ,  0.04618319, ...,  0.1345141 ,
         0.04746348, -0.05028719],
       [-0.09006829, -0.02984722,  0.03510597, ...,  0.0344655 ,
        -0.07471598,  0.04870407],
       [-0.02693345,  0.05441548,  0.0054687 , ...,  0.09743757,
        -0.13128257, -0.04685029],
       [ 0.00585679,  0.01583065,  0.02454555, ...,  0.11912089,
        -0.10991519,  0.0521571 ],
       [-0.08291283, -0.01857998,  0.03763821, ..., -0.02041946,
        -0.02709852,  0.04316133]], shape=(5, 384), dtype=float32)

In [12]:
len(student_text_embeddings)

25

In [14]:
student_text_embeddings.shape

(25, 384)

###  Build the FAISS index.


* Create an empty index — no data in it yet, just a container configured to hold vectors of a specific size.

* "Flat" means no approximation or clever indexing structure — it's brute-force, compares your query against every stored vector directly. 
  - Fine at 25 examples; but would need to change if we scaled to millions.

In [15]:
# Build the FAISS index.

# Create an empty index — no data in it yet, just a container configured to hold vectors of a specific size.

# .shape[1] retrieves the DIMENSIONALITY of each vector.
# IndexFlatL2(384) tells FAISS that we're going to give it vectors of length 384, and that when we search, compare them againt L2 distance(straight line distance).
index = faiss.IndexFlatL2(student_text_embeddings.shape[1])

In [16]:
# Takes the (25, 384) array and stores all 25 vectors inside the index, ready to be searched against.
index.add(student_text_embeddings)

* We now have a fully-built, searchable structure: 25 vectors stored, ready for index.search(query_vector, k) to compare a new QUERY VECTOR against all 25 and return the closest 'k' matches by distance. 

* Nothing has been searched yet at this point.
  - Only built the index.

In [17]:
print(f"Index built — {index.ntotal} vectors, dimension {student_text_embeddings.shape[1]}")

Index built — 25 vectors, dimension 384


In [18]:
# should be (N, 384) — N = however many examples we wrote, 384 = all-MiniLM-L6-v2's embedding dimension
print(student_text_embeddings.shape)  

(25, 384)


* retrieve_similar takes 2 inputs:
  - text (new sentence)

  - 'k' (how many similar exmaples to return). Here we're doing k=3 as a default.

* Converts our input text into a VECTOR, using the same "all-MiniLM-L6-v2" embedding model used to build the index.
  - Essential, as query has to live in the SAME 384-D vector space as everything stored in the index.

  - Else, distance comparisons would be meaningless.

  - EX: query_embedding = embedder.encode([text], convert_to_numpy=True)


* index.search() returns 2 things: 
  - distances (how far each match is).
  
  - indices (which stored vectors matched).

  - _, throws away the distances — since we'rwnot using them here, just grabbing which examples matched. 
  
  - INDICES comes back as a 2D array (shape (1, k), since we searched with one query) — a list of POSITIONS, like [[7, 2, 15]], NOT the actual text or vectors themselves.

* return [example_corpus[i] for i in indices[0]] 
  - List comprehension uses each integer 'i' as a lookup index into the original example_corpus list.
  
  - Converts "position 7 in the index" back into the actual dict we wrote by hand: {"student_text": "...", "emotions": [...], "teacher_response": "..."}. 
  
  - This only works correctly because example_corpus, corpus_texts, corpus_embeddings, and the FAISS index were all built in the same order — position 7 means the same example in all four.

In [29]:
def retrieve_similar(text, k=3):
  # Converts our input text into a VECTOR, using the same "all-MiniLM-L6-v2" embedding model used to build the index. 
  # Essential, as query has to live in the SAME 384-D vector space as everything stored in the index.
    query_embedding = embedder.encode([text], convert_to_numpy=True)
    
    # The search. FAISS compares query vector against all 25 stored vectors and finds the 'k' closest ones by L2 distance.
    _, indices = index.search(query_embedding, k)
    return [example_corpus[i] for i in indices[0]]

# Helper function to display results.
def display_results(results):
    for r in results:
        print(f"\n* {r['student_text']}")

In [33]:
test_text = example_corpus[0]["student_text"]
test_text

'I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.'

In [34]:
results = retrieve_similar(test_text, k=3)
display_results(results)


* I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.

* I don't understand why the experiment produced such a different result from what we expected. At first I thought we made a mistake, but now I'm wondering if there's something interesting happening that we haven't considered.

* I read the directions three times and still don't understand what we're supposed to include in the final section. I know I'm missing something, but I can't figure out what it is.


In [32]:
new_text = "I got a B+ on the essay but I know I could have done better if I'd started earlier"
results = retrieve_similar(new_text, k=3)
display_results(results)


* I studied almost every night this week because I wanted to improve my grade, but I got my test back today and it was barely higher than my last one. I thought all that work would make a bigger difference.

* My teacher stayed after class to help me understand the assignment even though she had other things to do. I really appreciate that she took the time to help me.

* I spent three hours working on the assignment and then found out the instructions had changed after I started. I wish someone had told us earlier because now I have to redo most of my work.


* Summary of Retriever.

  - Built dense retriever using sentence embeddings and FAISS at a smaller scale for the synthetic data.

  - Used learned embeddings from the "all-MiniLM-L6-v2: model to capture semantic meaning.

  - Used the FAISS vector index and its vector search library.

  - Embed the synthetic text corpus once.

  - Embed each query ONCE at search time, and then compare it against the vector space.

  - Return the top 'k' positions.


* Bigger Scale?
  - We'd switch off the brute-force IndexFlatL2 approach for an approximate index(IndexIVFFlat), that trades a small amount of accuracy for speed gains.

  - No chunking was done here, as our "documents" were small. 
  
  - Production RAG used chunking(fixed-size windows, semantic chunking, overlap handling) for longer documents.

  - Production RAG pipelines add a second-stage re-ranker(expensive but accurate model) that re-scores the top 'k' results from the fast retrieval.

  